# Practical 10 — Real-Estate Data Wrangling

Create real-estate data, standardize column names, fill missing values, convert dates, merge neighborhood data, filter, encode and remove outliers.

**Run the cells from top to bottom.** The dataset is generated inside this notebook before the practical begins.

## 1. Install / import requirements

If a package is missing, run `!pip install pandas numpy matplotlib seaborn scikit-learn statsmodels openpyxl requests xgboost` in a notebook cell. Most environments already have the common packages.

In [1]:
from pathlib import Path
Path('data').mkdir(exist_ok=True)


## 2. Create the required dataset

This is the relevant part of `make_data.py`, placed here so the notebook is self-contained.

In [2]:
# Create synthetic real-estate records with missing values and price outliers.
import numpy as np, pandas as pd
rng=np.random.default_rng(42); n=500
area=rng.integers(500,3500,n)
real_estate=pd.DataFrame({"Property ID":range(n),"Area (sqft)":area,
 "Bed rooms":rng.integers(1,6,n),"Property-Type":rng.choice(["House","Apartment","Villa"],n),
 "Neighborhood":rng.choice(["Kothrud","Baner","Hinjewadi","Wakad"],n),
 "Sale Date":pd.date_range("2022-01-01",periods=n,freq="2D").strftime("%Y-%m-%d"),
 "Sale Price $":(area*120+rng.normal(0,30000,n)).round()})
real_estate.loc[rng.choice(n,25,replace=False),"Area (sqft)"]=np.nan
real_estate.loc[rng.choice(n,5,replace=False),"Sale Price $"]*=15
real_estate.to_csv("data/RealEstate_Prices.csv",index=False)
print("Created data/RealEstate_Prices.csv")


Created data/RealEstate_Prices.csv


## 3. Practical implementation

The following cells perform the analysis and show the output.

In [3]:
import pandas as pd

df = pd.read_csv("data/RealEstate_Prices.csv")
# 1. clean column names: "Area (sqft)" -> "area_sqft"
df.columns = df.columns.str.lower().str.replace(r"[^a-z0-9]+", "_", regex=True).str.strip("_")
print(df.columns.tolist())

df["area_sqft"] = df.area_sqft.fillna(df.area_sqft.median())          # 2. missing values
df["sale_date"] = pd.to_datetime(df.sale_date)

# 3. merge with extra data (neighborhood info)
info = pd.DataFrame({"neighborhood": ["Kothrud", "Baner", "Hinjewadi", "Wakad"], "metro_km": [2, 5, 8, 7]})
df = df.merge(info, on="neighborhood", how="left")

# 4. filter / subset
sub = df[(df.sale_date >= "2023-01-01") & (df.property_type == "House")]
print("Houses sold since 2023:", len(sub))

df = pd.get_dummies(df, columns=["property_type"])                    # 5. encode categoricals
print(df.groupby("neighborhood").sale_price.mean().round())          # 6. aggregate

q1, q3 = df.sale_price.quantile([.25, .75])                           # 7. outliers (IQR)
df = df[df.sale_price.between(q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1))]
print("Rows after outlier removal:", len(df))

['property_id', 'area_sqft', 'bed_rooms', 'property_type', 'neighborhood', 'sale_date', 'sale_price']
Houses sold since 2023: 104
neighborhood
Baner        267329.0
Hinjewadi    238052.0
Kothrud      244966.0
Wakad        326302.0
Name: sale_price, dtype: float64
Rows after outlier removal: 496
